# Explore and save dataset

We'll follow these stepps to create the training dataset:

- **Collect Domain-Specific Documents**: Gather documents relevant to the domain you want to specialize the LLM in (e.g., medical documents for PubMed, legal documents, API documentation for software).
- **Chunk the file into Documents**
- For each Document chunk, generate a set of Questions that can be answered from the Document
- For each Document-Question pair, create a list of documents using:
  - Golden Document (D*): Document that contains the answer to the question.
  - Distractor Documents (Dk): Documents that do not contain relevant information.
- **Question-Answer-Document Triplets**: From each Document-Question pair, generate a factual Answer based on the Golden Document.
- **Add disctractor documents**
- **Save dataset**

In [ ]:
! pip install langchain langchain-community langchain-experimental langchain-openai pypdf

  Using cached langchain-0.3.7-py3-none-any.whl (1.0 MB)
     |████████████████████████████████| 2.4 MB 3.0 MB/s eta 0:00:01
     |████████████████████████████████| 208 kB 4.6 MB/s eta 0:00:01
  Using cached langchain_openai-0.2.6-py3-none-any.whl (50 kB)
  Using cached pypdf-5.1.0-py3-none-any.whl (297 kB)
  Using cached langchain_core-0.3.15-py3-none-any.whl (408 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
     |████████████████████████████████| 391 kB 9.8 MB/s eta 0:00:01
  Using cached numpy-1.26.4-cp39-cp39-macosx_11_0_arm64.whl (14.0 MB)
  Using cached langchain_text_splitters-0.3.2-py3-none-any.whl (25 kB)
     |████████████████████████████████| 306 kB 10.1 MB/s eta 0:00:01
     |████████████████████████████████| 2.1 MB 5.7 MB/s eta 0:00:01
  Using cached pydantic-2.9.2-py3-none-any.whl (434 kB)
  Using cached SQLAlchemy-2.0.35-cp39-cp39-macosx_11_0_arm64.whl (2.1 MB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
     |██████████████████████

### Import libraries

In [ ]:
import os
import random
from dotenv import load_dotenv
from langchain_experimental.text_splitter import SemanticChunker
from langchain.schema import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai.chat_models import ChatOpenAI

load_dotenv()

True

## Setting up the LLM

In [ ]:
model = ChatOpenAI(model_name="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

## Loading and chunking documents

In [ ]:

def load_and_chunk_pdf(pdf_path):
    """
    Load a PDF file and chunk it semantically using LangChain.

    Args:
        pdf_path (str): Path to the PDF file

    Returns:
        list: List of semantic chunks
    """
    # Initialize the PDF loader
    loader = PyPDFLoader(pdf_path)

    # Load the document
    pages = loader.load()


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=3000,
        chunk_overlap=500,
        length_function=len,
    is_separator_regex=False,
    )
    chunks = text_splitter.split_documents(pages)


    return chunks

In [ ]:
# calling the function to chunk the finance PDF
chunks = load_and_chunk_pdf("Indian_financial_system.pdf")


In [ ]:

print(f"Total number of chunks: {len(chunks)}")

# Print information about each chunk
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(f"Content length: {len(chunk.page_content)}")
    print(f"Metadata: {chunk.metadata}")
    print("-" * 50)
    if (i>5):
      break

Total number of chunks: 258

Chunk 1:
Content length: 1266
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 0}
--------------------------------------------------

Chunk 2:
Content length: 1859
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 1}
--------------------------------------------------

Chunk 3:
Content length: 1263
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 2}
--------------------------------------------------

Chunk 4:
Content length: 1655
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 3}
--------------------------------------------------

Chunk 5:
Content length: 1596
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 4}
--------------------------------------------------

Chunk 6:
Content length: 1882
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 5}
--------------------------------------------------

Chunk 7:
Content length: 1896
Metadata: {'source': 'Indian_financial_system.pdf', 'page': 6}
--------

In [ ]:
# checking contents of a random chunk
sample_index = random.randint(0, len(chunks)-1)
chunk = chunks[sample_index]
chunk

Document(metadata={'source': 'Indian_financial_system.pdf', 'page': 234}, page_content='purchases.  The barg ains that are struck in the trading ring by the \nmembers of the stock exchanges are at the fairest prices determined by \nthe basic laws of supply and demand. \nHistory of Stock Exchanges \nThe only stock exchange operating in the 19 th century were those of \nMumbai set up in 1875 and Ahamedabad set up in 1894.  These were \norganized as voluntary non -profit-making associations of brokers to \nregulate and protect their interests.  Before the control on securities \ntrading became a central subject under the Constitution in 1950, it was \na state subject and the Mumbai Securities Contracts (control) Act of \n1925 used to regulate trading in securities.  Under this Act, the \nMumbai Stock Exchange was recognized in 1927 and Ahamedabad in \n1937. During the was boom, a number of sto ck exchanges were \norganized even in Mumbai, Ahamedabad and other centres, but they \nwere not 

## Generate training data from the chunked documents



**The first step to generate data is by generating the questions for each chunk of the document using LLM**

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI
from typing import List, Dict, Any


# Generate questions for each document chunk
def generate_questions_for_chunk(chunk: str, model, num_questions: int = 3) -> List[str]:
    prompt_template = ChatPromptTemplate.from_template(
        template="Generate {num_questions} questions that can be answered based on the following text:\n\n{chunk}"
    )
    prompt = prompt_template.format(num_questions=num_questions, chunk=chunk)

    response = model.invoke(prompt)
    #print(response.content)
    questions = response.content.split("\n")[:num_questions]
    return [question.strip() for question in questions if question.strip()]


In [ ]:

# Main function to generate questions for all document chunks
def generate_questions_for_document(chunks: List[str], num_questions_per_chunk: int = 3) -> List[List[str]]:
    model = ChatOpenAI(model_name="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

    all_questions = []

    for i, chunk in enumerate(chunks):
        print(f"Generating questions for chunk {i + 1}...")
        data=chunk.page_content
        questions = generate_questions_for_chunk(data, model, num_questions_per_chunk)
        all_questions.append(questions)

    return all_questions

# Calling the function
questions_per_chunk = generate_questions_for_document(chunks, num_questions_per_chunk=3)

# Display questions for each chunk for understanding
for i, questions in enumerate(questions_per_chunk):
    print(f"\nQuestions for Chunk {i + 1}:")
    for q in questions:
        print(f"- {q}")
    if (i>5):
      break

Generating questions for chunk 1...
Generating questions for chunk 2...
Generating questions for chunk 3...
Generating questions for chunk 4...
Generating questions for chunk 5...
Generating questions for chunk 6...
Generating questions for chunk 7...
Generating questions for chunk 8...
Generating questions for chunk 9...
Generating questions for chunk 10...
Generating questions for chunk 11...
Generating questions for chunk 12...
Generating questions for chunk 13...
Generating questions for chunk 14...
Generating questions for chunk 15...
Generating questions for chunk 16...
Generating questions for chunk 17...
Generating questions for chunk 18...
Generating questions for chunk 19...
Generating questions for chunk 20...
Generating questions for chunk 21...
Generating questions for chunk 22...
Generating questions for chunk 23...
Generating questions for chunk 24...
Generating questions for chunk 25...
Generating questions for chunk 26...
Generating questions for chunk 27...
Generating

In [ ]:
len(questions_per_chunk)

### creating pairs and adding distractor documents

**The second step is for each of the chunk we have questions we wnat to create Document-Question pairs with Golden and Distractor documents**

so the function **create_document_question_pair**s creates pairs of questions with their correct (golden) documents and incorrect (distractor) documents

- For each chunk of text and its corresponding questions:

  - Extracts the actual text content from the chunk (golden_document_content)
  - Processes each question associated with that chunk
  - Creates a list of potential distractor documents by excluding the current chunk
  - Randomly samples the specified number of distractor documents




In [ ]:

# Function to create Document-Question pairs with Golden and Distractor documents
def create_document_question_pairs(chunks: List[Any], questions_per_chunk: List[List[str]], num_distractors: int = 2) -> List[Dict[str, Any]]:
    document_question_pairs = []

    for chunk_idx, chunk in enumerate(chunks):
        golden_document_content = chunk.page_content  # Extract the actual text content from the chunk

        # Generate a pair for each question associated with this chunk
        for question in questions_per_chunk[chunk_idx]:
            # Select distractor documents (randomly sample other chunks)
            potential_distractors = [c.page_content for i, c in enumerate(chunks) if i != chunk_idx]
            distractor_documents = random.sample(potential_distractors, min(num_distractors, len(potential_distractors)))

            # Create the Document-Question pair dictionary
            document_question_pair = {
                "question": question,
                "golden_document": golden_document_content,
                "distractor_documents": distractor_documents
            }

            document_question_pairs.append(document_question_pair)

    return document_question_pairs


Returns a list of dictionaries, where each dictionary contains:

- "question": The generated question
- "golden_document": The correct document chunk that answers the question
- "distractor_documents": Wrong document chunks that don't answer the question




In [ ]:
document_question_pairs = create_document_question_pairs(chunks, questions_per_chunk, num_distractors=3)


lets print and look at the structure in detail

In [ ]:

# Display some examples of Document-Question pairs
for i, pair in enumerate(document_question_pairs[:3]):
    print("--------------------------------------")
    print(f"\nDocument-Question Pair {i + 1}:")
    print(f"Question: {pair['question']}")
    print("--------------------------------------")
    print(f"Golden Document:\n{pair['golden_document']}")
    print("--------------------------------------")
    print(f"Distractor Documents:")
    for distractor in pair['distractor_documents']:
        print(f"- {distractor[:300]}...")  # Print a snippet of each distractor document not complete
        print("--------------------------------------")

--------------------------------------

Document-Question Pair 1:
Question: 1. What are the key components covered in the introduction to the Indian Financial System as outlined in the text?
--------------------------------------
Golden Document:
UNIT-I 
INDIAN FINANCIAL SYSTEM 
1. Introduction to Indian Financial System 
1.1 Significance and Definition 
1.2 Purpose and Organisation 
1.3 Liberalisation of the Financial System 
2. Saving and Financial Intermediation 
2.1 Savings 
2.2 Composition of Savings 
2.3 Factor Determining Savings 
2.4 Saving Rate in Ninth Plan 
2.5 Financial Liabilities 
2.6 Financial Intermediation 
3. Commercial Banking 
3.1 Evolution 
3.2 Financial Services 
3.3 Fiduciary Services 
3.4 Off Balance Sheet Activities 
3.5 Analysis of Assets and Liabilities of Schedule 
Commercial Banks 
4. Central Banking 
4.1 Introduction 
4.2 Instruments of Monetary Control 
4.3 Reserve Bank of India 
5. Public Depth 
5.1 Classification of Public Depth 
5.2 Secondary Depth Mar

### Adding answers to the triplet

**The next step is to the triplet of question-document-distractor that we have , we add the correct answers to it as well.**

The function **generate_answer** adds answer to our triplet and also formats everything before returning.



the flow of the function is as follows:

- Question and Context Extraction:

  - Extracts the question and the correct context (golden_document) from the input pair


- Prompt Construction:

  - Creates a detailed prompt with specific instructions to refer to context and answer in COT format also providing reasoning and ideas:

  - Message Structure:

    - SystemMessage: Defines the role as a helpful question assistant
    - HumanMessage: Contains the constructed prompt




- Answer Generation:

  - Sends the messages to the model
  - Captures the response content


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed # for parallel processing
from langchain.schema import SystemMessage, HumanMessage

# Generate answer for a single Document-Question pair
def generate_answer(document_question_pair: Dict[str, Any], model) -> Dict[str, Any]:
    question = document_question_pair["question"]
    context = document_question_pair["golden_document"]

    # Define the answer generation prompt
    prompt = f"""
    Question: {question}\n Context: {context}\n

    Answer this question using the information given in the context above and no other knowledge. Here are things to pay attention to:

    - First provide step-by-step reasoning on how you are answering the question and logic behind that based on the context.

    - In the reasoning, if you need to copy paste some sentences from the context, include them in ##begin_quote## and ##end_quote##.

    - End your response with final answer in the form <ANSWER>: $answer, the answer should be given in a friendly tone.

    - If the answer cannot be found in the context, say "I'm sorry, I cannot answer this question as I'm missing the required information."

    You MUST begin your final answer with the tag "<ANSWER>:".
    """

    # Construct message list
    messages = [
        SystemMessage(content="You are a helpful question assistant who can provide an answer given a question based on relevant context ."),
        HumanMessage(content=prompt)
    ]

    # Run the messages through the model
    response = model.invoke(messages)

    # Store the generated answer
    answer = response.content
    return {
        "question": question,
        "golden_document": context,
        "distractor_documents": document_question_pair["distractor_documents"],
        "answer": answer
    }


Returns a dictionary containing:

- Original question
- Golden document (correct context)
- Distractor documents
- Generated answer

This task is huge and will take a lot of time to complete.

it also depends a lot on the api side where we will keep on waiting for the response, so we can use multi threading to solve this issue and work faster.

The function **generate triplets in parallel**:
- Processes multiple document-question pairs simultaneously for faster execution
- Creates answer triplets (question, context, answer) in parallel

In [ ]:

# Main function to generate answers in parallel
def generate_triplets_in_parallel(document_question_pairs: List[Dict[str, Any]], num_threads: int = 5) -> List[Dict[str, Any]]:
    results = []

    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        future_to_pair = {executor.submit(generate_answer, pair, model): pair for pair in document_question_pairs}

        for future in as_completed(future_to_pair): # Uses as_completed to process results as they finish
            result = future.result()
            results.append(result)
            print(f"Processed {len(results)}/{len(document_question_pairs)} pairs")

    return results

triplets = generate_triplets_in_parallel(document_question_pairs, num_threads=5)

Processed 1/671 pairs
Processed 2/671 pairs
Processed 3/671 pairs
Processed 4/671 pairs
Processed 5/671 pairs
Processed 6/671 pairs
Processed 7/671 pairs
Processed 8/671 pairs
Processed 9/671 pairs
Processed 10/671 pairs
Processed 11/671 pairs
Processed 12/671 pairs
Processed 13/671 pairs
Processed 14/671 pairs
Processed 15/671 pairs
Processed 16/671 pairs
Processed 17/671 pairs
Processed 18/671 pairs
Processed 19/671 pairs
Processed 20/671 pairs
Processed 21/671 pairs
Processed 22/671 pairs
Processed 23/671 pairs
Processed 24/671 pairs
Processed 25/671 pairs
Processed 26/671 pairs
Processed 27/671 pairs
Processed 28/671 pairs
Processed 29/671 pairs
Processed 30/671 pairs
Processed 31/671 pairs
Processed 32/671 pairs
Processed 33/671 pairs
Processed 34/671 pairs
Processed 35/671 pairs
Processed 36/671 pairs
Processed 37/671 pairs
Processed 38/671 pairs
Processed 39/671 pairs
Processed 40/671 pairs
Processed 41/671 pairs
Processed 42/671 pairs
Processed 43/671 pairs
Processed 44/671 pai

In [ ]:

# Displaying a few triplets
for i, triplet in enumerate(triplets[:3]):
    print("--------------------------------------")
    print(f"\nTriplet {i + 1}:")
    print(f"Question: {triplet['question']}")
    print("--------------------------------------")
    print(f"Answer: {triplet['answer']}")
    print("--------------------------------------")
    print(f"Golden Document:\n{triplet['golden_document']}")
    print("--------------------------------------")
    print(f"Distractor Documents:")
    for distractor in triplet['distractor_documents']:
        print(f"- {distractor[:300]}...")  # Print a snippet of each distractor document for brevity
        print("--------------------------------------")

--------------------------------------

Triplet 1:
Question: 2. How has the liberalisation of the financial system impacted the Indian economy according to the text?
--------------------------------------
Answer: To answer the question about how the liberalisation of the financial system has impacted the Indian economy, we can follow these steps:

1. Identify the key term in the question: "liberalisation of the financial system."
2. Look for any mentions of this term in the provided context.
3. Focus on the sections that discuss the significance of the financial system and any changes that have occurred due to liberalisation.
4. Extract relevant information that illustrates the effects of this liberalisation on the Indian economy.

Upon examining the context, we see that it mentions, "the economic scene in the post independence period has seen a sea change; the end result being that the economy has made enormous progress in diverse fields." This suggests that the liberalisation of the 

In [ ]:
triplets[0]

{'question': '2. How has the liberalisation of the financial system impacted the Indian economy according to the text?',
 'golden_document': 'UNIT-I \nINDIAN FINANCIAL SYSTEM \n1. Introduction to Indian Financial System \n1.1 Significance and Definition \n1.2 Purpose and Organisation \n1.3 Liberalisation of the Financial System \n2. Saving and Financial Intermediation \n2.1 Savings \n2.2 Composition of Savings \n2.3 Factor Determining Savings \n2.4 Saving Rate in Ninth Plan \n2.5 Financial Liabilities \n2.6 Financial Intermediation \n3. Commercial Banking \n3.1 Evolution \n3.2 Financial Services \n3.3 Fiduciary Services \n3.4 Off Balance Sheet Activities \n3.5 Analysis of Assets and Liabilities of Schedule \nCommercial Banks \n4. Central Banking \n4.1 Introduction \n4.2 Instruments of Monetary Control \n4.3 Reserve Bank of India \n5. Public Depth \n5.1 Classification of Public Depth \n5.2 Secondary Depth Market \n5.3 Repos  \n5.4 Reverse Repo \n6. Advances to Priority Sector \n7. Super

## Exporting the dataset
now we transforms the triplets (question-answer-document sets) into a structured DataFrame format

In [ ]:
import pandas as pd

data = []
for triplet in triplets: # creates a dictionary for each and appends it to create a df
    data.append({
        "question_string": triplet["question"],
        "answer_string": triplet["answer"],
        "golden_document_string": triplet["golden_document"],
        "distractor_document_string": " ||| ".join(triplet["distractor_documents"])  # Concatenate distractor documents with separator
    })

triplets_df = pd.DataFrame(data)



In [ ]:
triplets_df.head()

,question_string,answer_string,golden_document_string,distractor_document_string
0,2. How has the liberalisation of the financial...,To answer the question about how the liberalis...,UNIT-I \nINDIAN FINANCIAL SYSTEM \n1. Introduc...,No. that have eroded \nnet worth \n139 139 144...
1,1. What are the key components covered in the ...,To answer the question about the key component...,UNIT-I \nINDIAN FINANCIAL SYSTEM \n1. Introduc...,impact of maturing debt on the foreign exchang...
2,1. What is the primary function of the financi...,To answer the question about the primary funct...,1980s have led to the conclusion that to obtai...,programme of economic liberalisation. While th...
3,3. What are some components included in the de...,To answer the question about the components in...,1980s have led to the conclusion that to obtai...,to apply for category I status or take up some...
4,2. How does the financial system contribute to...,To answer the question about how the financial...,1980s have led to the conclusion that to obtai...,"and financial instituti ons, whose operations ..."


In [ ]:
triplets_df.to_csv("triplet.csv", index=False)

In [ ]:
triplets_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 671 entries, 0 to 670
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   question_string             671 non-null    object
 1   answer_string               671 non-null    object
 2   golden_document_string      671 non-null    object
 3   distractor_document_string  671 non-null    object
dtypes: object(4)
memory usage: 21.1+ KB


the above is for our convinience and for us to be able to make any changes when required, but azure requires the dataset to be in jsonl format

### JSONL

JSONL (JSON Lines) is a text-based file format that stores structured data as a collection of JSON objects, one per line. It's similar to JSON, but uses newline characters to separate JSON values.

refer to https://learn.microsoft.com/en-us/azure/ai-studio/how-to/fine-tune-model-llama?tabs=llama-three%2Cchatcompletion for more infor on the data format required for finetuning

In [ ]:
import json

# Define the system message content
system_message = {"role": "system", "content": "answer the question given by the user on the basis of only relevant content from the context documents provided "}

# Convert each row to JSON Lines format
jsonl_data = []
for _, row in triplets_df.iterrows():
    # Construct user message with markdown formatting
    user_content = (
        f"**Question**: {row['question_string']}\n\n"
        f"**Document**: {row['golden_document_string']}\n\n"
        f"**Document**: {row['distractor_document_string']}"
    )
    user_message = {"role": "user", "content": user_content}

    # Construct assistant message
    assistant_message = {"role": "assistant", "content": row["answer_string"]}

    # Create the full message structure for the row
    conversation = {
        "messages": [system_message, user_message, assistant_message]
    }
    jsonl_data.append(conversation)

# Write to a .jsonl file
with open("output.jsonl", "w") as f:
    for entry in jsonl_data:
        json.dump(entry, f)
        f.write("\n")